# PQID — Metadata Enrichment

Executes each entry's Qiskit code, extracts circuit-level metadata, and writes enriched JSONL files.

**⚠️ Kernel**: must be Python 3.11 — the default kernel does NOT have Qiskit installed.
Set the kernel to `Python 3.11` (or `torch311env`) before running.

| Cell | Purpose |
|------|---------|
| 2 | Imports, paths, constants |
| 3 | Size / expressiveness / difficulty classifiers |
| 4 | Feature extractors (topology, transpilation, gate profile, …) |
| 5 | Benchmark diagnostics (deprecated API, hallucination type) |
| 6 | Execution engine (namespace, timeout exec, validation status) |
| 7 | Stats extraction + per-entry enrichment |
| 8 | I/O helpers + resume cache |
| 9 | Load cache (run once before any split cell) |
| 10 | Process `train_clean.jsonl` |
| 11 | Process `validation_clean.jsonl` |
| 12 | Process `test_clean.jsonl` |
| 13 | Atomic rename + final summary |

**Resume-safe**: each entry is written to `enrich_metadata_cache.jsonl` immediately after processing.
Interrupt any split cell and re-run — cached entries are restored without re-executing Qiskit code.

**Output**: enriched `*_clean.jsonl` files in `PQID/data/processed/`.

## Cell 2 — Imports & Config

In [ ]:
import json
import os
import threading
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PQID" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
BASE = str(PROJECT_ROOT / "data" / "processed")

SPLITS = {
    "train": {
        "input": f"{BASE}/train_clean.jsonl",
        "tmp":   f"{BASE}/train_clean_enriched.jsonl",
    },
    "validation": {
        "input": f"{BASE}/validation_clean.jsonl",
        "tmp":   f"{BASE}/validation_clean_enriched.jsonl",
    },
    "test": {
        "input": f"{BASE}/test_clean.jsonl",
        "tmp":   f"{BASE}/test_clean_enriched.jsonl",
    },
}

# Per-entry resume cache
CACHE_FILE = f"{BASE}/enrich_metadata_cache.jsonl"

# ---------------------------------------------------------------------------
# Gate-set constants
# ---------------------------------------------------------------------------
CLIFFORD_GATES = {
    "h", "cx", "cy", "cz", "x", "y", "z",
    "s", "sdg", "swap", "iswap", "ecr",
}
UNIVERSAL_GATES = {"t", "tdg", "rz", "rx", "ry", "u", "u1", "u2", "u3", "ccx"}
ROTATION_GATES = {
    "rx", "ry", "rz", "r", "p", "u", "u1", "u2", "u3",
    "rxx", "ryy", "rzz", "rzx", "xx_minus_yy", "xx_plus_yy",
}
ENTANGLING_GATE_NAMES = {
    "cx", "cy", "cz", "ch", "cp", "cu", "cu1", "cu2", "cu3",
    "swap", "iswap", "dcx", "ecr",
    "ccx", "ccz", "cswap", "c3x", "c4x",
}
STANDARD_GATES = (
    CLIFFORD_GATES | UNIVERSAL_GATES | ROTATION_GATES | ENTANGLING_GATE_NAMES
    | {"id", "sx", "sxdg", "x", "y", "z", "measure", "barrier", "reset", "delay"}
)
SIZE_CLASSES = ["trivial", "simple", "moderate", "complex", "very_complex"]

# Transpilation target: IBM standard basis, no coupling map
TRANSPILE_BASIS_GATES = ["cx", "rz", "sx", "x"]

# Deprecated Qiskit API patterns (text-level heuristic)
DEPRECATED_PATTERNS = [
    "execute(",
    "Aer.get_backend(",
    "BasicAer",
]

# Enrichment-computed fields stored in the cache record
_ENRICHMENT_FIELDS = frozenset({
    "validation_status", "validation_error_type", "circuit_stats_available",
    "num_qubits", "num_clbits", "gate_count", "circuit_depth", "circuit_width",
    "gate_types", "num_gate_types", "avg_gates_per_layer", "has_measurement",
    "is_parameterized", "t_count", "t_depth",
    "circuit_expressiveness", "size_class", "benchmark_difficulty",
    "has_clifford_only", "has_clifford_t", "has_rotation_gates",
    "has_entangling_gates", "has_barriers", "has_custom_gates",
    "two_qubit_gate_count", "entangling_gate_ratio",
    "num_parameters", "parameter_density", "parameter_reuse",
    "measurement_count", "reset_usage", "mid_circuit_measurement",
    "classical_register_count",
    "interaction_graph_edges", "graph_density", "max_qubit_degree",
    "connected_components",
    "transpiled_depth", "transpiled_gate_count", "transpiled_cx_count",
    "transpiled_single_qubit_count", "transpilation_overhead",
    "transpilation_successful",
    "prompt_word_count", "prompt_length_chars",
    "openqasm3_export_successful", "openqasm3_export_error",
    "qiskit_version", "transpilation_basis_gates",
    "prompt_token_count_cl100k",
    "code_lines",
    "is_unitary", "gate_set_diversity", "entanglement_depth",
    "unconnected_qubit_count",
    "api_deprecated_usage", "deprecated_api_patterns", "hallucination_type",
    "transpilation_depth_ratio",
})

# ---------------------------------------------------------------------------
# Qiskit imports
# ---------------------------------------------------------------------------
print("Importing Qiskit...", flush=True)
import numpy as np
from numpy import pi
import math

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile as qk_transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import (
    PermutationGate,
    XGate, YGate, ZGate, HGate, SGate, TGate,
    CXGate, CZGate, CCXGate, SwapGate,
    RXGate, RYGate, RZGate,
    UGate, U1Gate, U2Gate, U3Gate,
)
import qiskit
QISKIT_VERSION = qiskit.__version__

print(f"Qiskit {QISKIT_VERSION} ready.", flush=True)
print(f"BASE      : {BASE}")
print(f"CACHE     : {CACHE_FILE}")
for name, paths in SPLITS.items():
    exists = os.path.exists(paths['input'])
    print(f"  {name:<12}: {paths['input']}  {'✓' if exists else '✗ NOT FOUND'}")

## Cell 3 — Classifiers

Pure functions: circuit expressiveness, size class, benchmark difficulty.

In [ ]:
def classify_expressiveness(gate_types: dict, is_parameterized: bool) -> str:
    """
    Return quantum computational expressiveness class.
      'parameterized' — circuit has free parameters (variational / PQC)
      'universal'     — contains T/Tdg or continuous rotation gates
      'clifford'      — all gates are Clifford; efficiently classically simulable
    """
    if is_parameterized:
        return "parameterized"
    gates_used = set(gate_types.keys())
    if gates_used & UNIVERSAL_GATES:
        return "universal"
    return "clifford"


def classify_size(num_qubits: int, circuit_depth: int, gate_count: int) -> str:
    """
    Return size class based on the maximum level implied by any of the three metrics.

    trivial     : qubits ≤ 2   | depth ≤ 2   | gates ≤ 3
    simple      : qubits 3–5   | depth 3–10  | gates 4–20
    moderate    : qubits 6–10  | depth 11–30 | gates 21–60
    complex     : qubits 11–20 | depth 31–80 | gates 61–200
    very_complex: qubits ≥ 21  | depth ≥ 81  | gates ≥ 201
    """
    def _qubit_level(n): return 0 if n<=2 else 1 if n<=5 else 2 if n<=10 else 3 if n<=20 else 4
    def _depth_level(d): return 0 if d<=2 else 1 if d<=10 else 2 if d<=30 else 3 if d<=80 else 4
    def _gate_level(g):  return 0 if g<=3 else 1 if g<=20 else 2 if g<=60 else 3 if g<=200 else 4
    return SIZE_CLASSES[max(_qubit_level(num_qubits), _depth_level(circuit_depth), _gate_level(gate_count))]


def compute_benchmark_difficulty(
    size_class: str, circuit_expressiveness: str,
    entangling_gate_ratio: float, num_parameters: int,
) -> str:
    """
    Composite benchmark_difficulty: easy / medium / hard.

    Score contributions (range 0–10):
      size_class (0–4) + expressiveness (0–2) +
      entangling ratio (0–2) + parameter count (0–2)
    Thresholds: easy ≤ 3 | medium 4–7 | hard ≥ 8
    """
    size_score  = {"trivial": 0, "simple": 1, "moderate": 2, "complex": 3, "very_complex": 4}.get(size_class, 2)
    expr_score  = {"clifford": 0, "parameterized": 1, "universal": 2}.get(circuit_expressiveness, 1)
    ent_score   = 0 if entangling_gate_ratio < 0.1 else 1 if entangling_gate_ratio < 0.3 else 2
    param_score = 0 if num_parameters == 0 else 1 if num_parameters <= 5 else 2
    total = size_score + expr_score + ent_score + param_score
    return "easy" if total <= 3 else "medium" if total <= 7 else "hard"


print("Classifiers defined.")

## Cell 4 — Feature Extractors

All `compute_*` functions that derive metadata from a live `QuantumCircuit` object.

In [ ]:
def _iter_instructions(qc):
    """Yield (operation, qubits, clbits) — compatible with Qiskit < 0.45 and ≥ 0.45."""
    for item in qc.data:
        if hasattr(item, "operation"):
            yield item.operation, item.qubits, item.clbits
        else:
            yield item[0], item[1], item[2]


def compute_entanglement_features(qc) -> dict:
    two_q = sum(1 for _, qubits, _ in _iter_instructions(qc) if len(qubits) >= 2)
    total = qc.size()
    return {
        "two_qubit_gate_count":  two_q,
        "entangling_gate_ratio": round(two_q / total, 4) if total > 0 else 0.0,
    }


def compute_parameterization_features(qc) -> dict:
    params    = list(qc.parameters)
    n_params  = len(params)
    density   = round(n_params / qc.num_qubits, 4) if qc.num_qubits > 0 else 0.0
    param_use_count: dict = {}
    for op, _, _ in _iter_instructions(qc):
        for p in op.params:
            if hasattr(p, "name"):
                key = str(p)
                param_use_count[key] = param_use_count.get(key, 0) + 1
    return {
        "num_parameters":    n_params,
        "parameter_density": density,
        "parameter_reuse":   any(v > 1 for v in param_use_count.values()),
    }


def compute_measurement_features(qc, gate_types: dict) -> dict:
    reset_usage = "reset" in gate_types
    mid_circuit = False
    seen_measure = False
    for op, _, _ in _iter_instructions(qc):
        if op.name == "measure":
            seen_measure = True
        elif seen_measure and op.name != "barrier":
            mid_circuit = True
            break
    return {
        "measurement_count":        gate_types.get("measure", 0),
        "reset_usage":              reset_usage,
        "mid_circuit_measurement":  mid_circuit,
        "classical_register_count": len(qc.cregs),
    }


def compute_gate_profile(gate_types: dict) -> dict:
    gates    = set(gate_types.keys())
    non_meta = gates - {"barrier", "measure", "reset", "delay", "id"}
    return {
        "has_clifford_only":    bool(non_meta) and non_meta.issubset(CLIFFORD_GATES),
        "has_clifford_t":       bool(gates & {"t", "tdg"}),
        "has_rotation_gates":   bool(gates & ROTATION_GATES),
        "has_entangling_gates": bool(gates & ENTANGLING_GATE_NAMES),
        "has_barriers":         "barrier" in gates,
        "has_custom_gates":     bool(gates - STANDARD_GATES),
    }


def compute_unconnected_qubit_count(qc) -> int:
    """Count qubits declared but never referenced by any gate."""
    used = set()
    for _, qubits, _ in _iter_instructions(qc):
        for q in qubits:
            used.add(q)
    return max(0, qc.num_qubits - len(used))


def compute_topology_features(qc) -> dict:
    """Interaction graph metrics via networkx. Returns None values if networkx absent."""
    _null = {"interaction_graph_edges": None, "graph_density": None,
             "max_qubit_degree": None, "connected_components": None}
    try:
        import networkx as nx
    except ImportError:
        return _null
    n = qc.num_qubits
    if n == 0:
        return {k: 0 for k in _null}
    G = nx.Graph()
    G.add_nodes_from(range(n))
    qubit_index = {q: i for i, q in enumerate(qc.qubits)}
    for _, qubits, _ in _iter_instructions(qc):
        idx = [qubit_index[q] for q in qubits]
        if len(idx) >= 2:
            for i in range(len(idx)):
                for j in range(i + 1, len(idx)):
                    G.add_edge(idx[i], idx[j])
    edges      = G.number_of_edges()
    max_poss   = n * (n - 1) / 2
    density    = round(edges / max_poss, 4) if max_poss > 0 else 0.0
    degrees    = [d for _, d in G.degree()]
    return {
        "interaction_graph_edges": edges,
        "graph_density":           density,
        "max_qubit_degree":        max(degrees) if degrees else 0,
        "connected_components":    nx.number_connected_components(G),
    }


def compute_transpilation_depth_ratio(transpiled_depth, circuit_depth) -> float:
    """transpiled_depth / circuit_depth. None if transpilation failed or depth is 0."""
    if transpiled_depth is None or circuit_depth is None or circuit_depth == 0:
        return None
    return round(transpiled_depth / circuit_depth, 4)


def compute_transpilation_features(qc) -> dict:
    """
    Transpile to IBM basis (cx/rz/sx/x), optimization_level=1, no coupling map.
    Backend-agnostic logical hardware cost.
    """
    _null = {
        "transpiled_depth":               None,
        "transpiled_gate_count":          None,
        "transpiled_cx_count":            None,
        "transpiled_single_qubit_count":  None,
        "transpilation_overhead":         None,
        "transpilation_successful":       False,
        "transpilation_basis_gates":      TRANSPILE_BASIS_GATES,
        "transpilation_depth_ratio":      None,
    }
    try:
        tqc         = qk_transpile(qc, basis_gates=TRANSPILE_BASIS_GATES, optimization_level=1)
        gate_counts = tqc.count_ops()
        cx_count    = gate_counts.get("cx", 0)
        total       = sum(gate_counts.values())
        orig_total  = qc.size()
        overhead    = round(total / orig_total, 4) if orig_total > 0 else None
        _t_depth    = tqc.depth()
        return {
            "transpiled_depth":               _t_depth,
            "transpiled_gate_count":          total,
            "transpiled_cx_count":            cx_count,
            "transpiled_single_qubit_count":  total - cx_count,
            "transpilation_overhead":         overhead,
            "transpilation_successful":       True,
            "transpilation_basis_gates":      TRANSPILE_BASIS_GATES,
            "transpilation_depth_ratio":      compute_transpilation_depth_ratio(_t_depth, qc.depth()),
        }
    except Exception:
        return _null


print("Feature extractors defined.")

## Cell 5 — Benchmark Diagnostics

Text-level deprecated-API detection, hallucination classification — no Qiskit execution needed.

In [ ]:
def detect_deprecated_api_usage(output_code: str) -> tuple:
    """
    Scan raw code for deprecated Qiskit API patterns (text-level, no exec).

    Returns (api_deprecated_usage, deprecated_api_patterns):
      bool | None   — True if any pattern matched
      list | None   — matched pattern strings; empty list if none matched
    Returns (None, None) when output_code is empty.
    """
    if not output_code or not output_code.strip():
        return None, None
    matched = [p for p in DEPRECATED_PATTERNS if p in output_code]
    return bool(matched), matched


def classify_hallucination_type(
    validation_status: str,
    validation_error_type: str,
    error_text: str = None,
) -> str:
    """
    Map validation outcome → benchmark-grade diagnostic failure category.

    Values: none | timeout | syntax_failure | dependency_hallucination |
            symbol_resolution_failure | non_circuit_execution |
            register_index_error | api_hallucination | runtime_semantic_failure

    Returns None if validation_status is None or unrecognised.
    """
    if not validation_status:
        return None
    if validation_status == "validated":
        return "none"
    if validation_status == "timeout":
        return "timeout"
    if validation_status == "syntax_error":
        return "syntax_failure"
    if validation_status == "import_error":
        return "dependency_hallucination"
    if validation_status == "name_error":
        return "symbol_resolution_failure"
    if validation_status == "no_circuit":
        return "non_circuit_execution"
    if validation_status == "exec_error":
        combined = " ".join(filter(None, [
            (validation_error_type or "").lower(),
            (error_text or "").lower(),
        ]))
        if any(kw in combined for kw in ("indexerror", "index", "registerror")):
            return "register_index_error"
        if any(kw in combined for kw in ("attributeerror", "typeerror")):
            return "api_hallucination"
        return "runtime_semantic_failure"
    return None


print("Benchmark diagnostics defined.")

## Cell 6 — Execution Engine

Exec namespace (pre-populated variables), timeout-guarded execution, validation status classification, circuit extraction from namespace.

In [ ]:
def make_namespace() -> dict:
    """Return a fresh exec() globals dict with common Qiskit objects pre-bound."""
    n = 3
    ns = {
        # Standard library
        "np": np, "pi": pi, "math": math,
        # Qiskit core
        "QuantumCircuit": QuantumCircuit, "QuantumRegister": QuantumRegister,
        "ClassicalRegister": ClassicalRegister,
        "Parameter": Parameter, "ParameterVector": ParameterVector,
        # Gate classes
        "PermutationGate": PermutationGate,
        "XGate": XGate, "YGate": YGate, "ZGate": ZGate, "HGate": HGate,
        "SGate": SGate, "TGate": TGate,
        "CXGate": CXGate, "CZGate": CZGate, "CCXGate": CCXGate, "SwapGate": SwapGate,
        "RXGate": RXGate, "RYGate": RYGate, "RZGate": RZGate,
        "UGate": UGate, "U1Gate": U1Gate, "U2Gate": U2Gate, "U3Gate": U3Gate,
        # Common size / index variables
        "n": n, "num_qubits": n, "nqubits": n, "N_QUBITS": n,
        "nq": n, "n_qubits": n, "num_q": n, "nQubits": n,
        "Nwires": n, "N": n, "nb_qubits": n, "nb": n,
        "num_trash": 1, "num_latent": 1,
        "var_qubits": n, "output_qubit": n, "qubit": n,
        "REAL_DIST_NQUBITS": n, "number_state": n, "qN": n,
        # Common angle variables
        "theta": pi/4, "phi": pi/2, "lam": pi/3,
        "alpha": pi/4, "beta": pi/4, "gamma": pi/4, "delta": pi/4,
        "angle": pi/4, "phase": pi/4, "omega": 2*pi/8,
        # Math shortcuts (circuits that omit np. prefix)
        "sin": np.sin, "cos": np.cos, "exp": np.exp, "sqrt": np.sqrt,
        # Common scalar variables
        "a": 0.5, "b": 0.5, "t": 0.5, "u": 0.5,
        # Common list variables
        "betas":  [pi/4]*n, "gammas": [pi/4]*n, "params": [pi/4]*n,
        # Pre-built registers
        "q":      QuantumRegister(n, "q"),   "qr":    QuantumRegister(n, "qr"),
        "cr":     ClassicalRegister(n, "cr"),"c":     ClassicalRegister(n, "c"),
        "cq":     ClassicalRegister(n, "cq"),"creg":  ClassicalRegister(n, "creg"),
        "qreg":   QuantumRegister(n, "qreg"),"qreg2": QuantumRegister(n, "qreg2"),
        "creg2":  ClassicalRegister(n, "creg2"),
        "qr_state": QuantumRegister(n, "qr_state"),
        "qr_obj":   QuantumRegister(1, "qr_obj"),
        "i_q":  QuantumRegister(n, "i_q"),  "i_c":  ClassicalRegister(n, "i_c"),
        "ii_q": QuantumRegister(n, "ii_q"), "ii_c": ClassicalRegister(n, "ii_c"),
        "q1": QuantumRegister(n, "q1"), "q2": QuantumRegister(n, "q2"),
        "q3": QuantumRegister(n, "q3"),
        "qa": QuantumRegister(n, "qa"),  "qb": QuantumRegister(n, "qb"),
        "qcar": QuantumRegister(1, "qcar"), "lq": QuantumRegister(n, "lq"),
        "sb":    QuantumRegister(1, "sb"),
        "ebit0": QuantumRegister(1, "ebit0"), "ebit1": QuantumRegister(1, "ebit1"),
        "c3":   ClassicalRegister(3, "c3"),
        "crx":  ClassicalRegister(n, "crx"), "crz": ClassicalRegister(n, "crz"),
        "qtemp":  QuantumRegister(n, "qtemp"), "qNtemp": QuantumRegister(n, "qNtemp"),
        "anc":     QuantumRegister(1, "anc"),
        "ancilla": QuantumRegister(1, "ancilla"),
        "aux":     QuantumRegister(1, "aux"),
        "work":    QuantumRegister(1, "work"),
        "flag":    QuantumRegister(1, "flag"),
        # Pre-built circuits (overwritten by most code)
        "qc":   QuantumCircuit(QuantumRegister(n, "q"), ClassicalRegister(n, "cr")),
        "qc2":  QuantumCircuit(QuantumRegister(n, "q"), ClassicalRegister(n, "cr")),
        "circ": QuantumCircuit(n),
        "ans":  QuantumCircuit(n),
        # Loop / index variables
        "i": 0, "j": 0, "k": 0, "m": 0,
        "G": np.eye(4), "pair": (0, 1),
    }
    return ns


def exec_with_timeout(code: str, timeout: float = 3.0):
    """
    Execute code in a fresh namespace with a hard timeout (daemon thread).
    Returns (namespace_dict, None) on success or (None, error_str) on failure/timeout.
    """
    result_box = [None]
    error_box  = [None]

    def _run():
        ns = make_namespace()
        try:
            exec(compile(code, "<string>", "exec"), ns)  # noqa: S102
            result_box[0] = ns
        except Exception as exc:
            error_box[0] = repr(exc)

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        return None, "TimeoutError: execution exceeded 3s"
    if result_box[0] is not None:
        return result_box[0], None
    return None, error_box[0] or "UnknownError"


def classify_validation_status(err, has_circuit: bool) -> tuple:
    """
    Returns (validation_status, validation_error_type).
    Possible statuses: validated | timeout | no_circuit | import_error |
                       name_error | syntax_error | exec_error
    """
    if err is None and has_circuit:
        return "validated", ""
    if err is None and not has_circuit:
        return "no_circuit", ""
    if "TimeoutError" in err:
        return "timeout", "TimeoutError"
    error_type = err.split("(")[0].split(".")[-1] if err else "UnknownError"
    if "ImportError" in err or "ModuleNotFoundError" in err:
        return "import_error", error_type
    if "NameError" in err:
        return "name_error", error_type
    if "SyntaxError" in err:
        return "syntax_error", error_type
    return "exec_error", error_type


def extract_best_circuit(ns: dict):
    """
    Find all QuantumCircuit objects in the namespace and return the largest one
    (by num_qubits * depth). Also searches one level inside lists/tuples.
    Returns None if no QuantumCircuit found.
    """
    circuits = [v for v in ns.values() if isinstance(v, QuantumCircuit)]
    for v in ns.values():
        if isinstance(v, (list, tuple)):
            for item in v:
                if isinstance(item, QuantumCircuit) and item not in circuits:
                    circuits.append(item)
    if not circuits:
        return None
    if len(circuits) == 1:
        return circuits[0]
    def score(qc):
        d = qc.depth()
        return qc.num_qubits * (d if d > 0 else 1)
    circuits.sort(key=score)
    return circuits[-1]


print("Execution engine defined.")

## Cell 7 — Stats Extraction & Per-Entry Enrichment

`extract_circuit_stats` assembles all metrics from a live `QuantumCircuit`.
`enrich_entry` orchestrates exec → validate → stats → metadata.

In [ ]:
def extract_circuit_stats(qc: QuantumCircuit) -> dict:
    """Pull all metadata from a QuantumCircuit. Returns a flat dict."""
    gate_types       = {k: int(v) for k, v in qc.count_ops().items()}
    has_measurement  = "measure" in gate_types
    is_parameterized = len(qc.parameters) > 0
    t_count          = gate_types.get("t", 0) + gate_types.get("tdg", 0)
    num_gate_types   = len(gate_types)
    _depth           = qc.depth()
    _size            = qc.size()
    avg_gates_per_layer = round(_size / _depth, 4) if _depth > 0 else 0.0

    try:
        t_depth = qc.depth(filter_function=lambda x: x.operation.name in {"t", "tdg"})
    except Exception:
        t_depth = None
    try:
        entanglement_depth = qc.depth(filter_function=lambda x: len(x.qubits) >= 2)
    except Exception:
        entanglement_depth = None

    # Shannon entropy of gate-type distribution (bits)
    _counts = [v for v in gate_types.values() if v > 0]
    _total  = sum(_counts)
    if _total > 0 and len(_counts) > 1:
        gate_set_diversity = round(
            -sum((c/_total) * math.log2(c/_total) for c in _counts), 4
        )
    else:
        gate_set_diversity = 0.0

    circuit_expressiveness = classify_expressiveness(gate_types, is_parameterized)
    size_class             = classify_size(qc.num_qubits, _depth, _size)
    entanglement           = compute_entanglement_features(qc)
    params_meta            = compute_parameterization_features(qc)
    measure_meta           = compute_measurement_features(qc, gate_types)
    gate_profile           = compute_gate_profile(gate_types)
    topology               = compute_topology_features(qc)
    transpilation          = compute_transpilation_features(qc)

    is_unitary = (
        not measure_meta["measurement_count"]
        and not measure_meta["reset_usage"]
    )
    difficulty = compute_benchmark_difficulty(
        size_class, circuit_expressiveness,
        entanglement["entangling_gate_ratio"],
        params_meta["num_parameters"],
    )

    return {
        # Core circuit metrics
        "num_qubits":              qc.num_qubits,
        "num_clbits":              qc.num_clbits,
        "gate_count":              _size,
        "circuit_depth":           _depth,
        "circuit_width":           qc.width(),
        "unconnected_qubit_count": compute_unconnected_qubit_count(qc),
        "gate_types":              gate_types,
        "num_gate_types":          num_gate_types,
        "avg_gates_per_layer":     avg_gates_per_layer,
        "has_measurement":         has_measurement,
        "is_parameterized":        is_parameterized,
        "t_count":                 t_count,
        "t_depth":                 t_depth,
        # XAI complexity
        "circuit_expressiveness":  circuit_expressiveness,
        "size_class":              size_class,
        "benchmark_difficulty":    difficulty,
        # Gate-set profile
        **gate_profile,
        "is_unitary":              is_unitary,
        "gate_set_diversity":      gate_set_diversity,
        # Entanglement
        **entanglement,
        "entanglement_depth":      entanglement_depth,
        # Parameterization
        **params_meta,
        # Measurement / output structure
        **measure_meta,
        # Topology
        **topology,
        # Transpilation (includes transpilation_depth_ratio)
        **transpilation,
        # Sentinel
        "circuit_stats_available": True,
    }


def enrich_entry(entry: dict) -> dict:
    """
    Execute entry['output'] and append all circuit metadata.
    Always returns an entry — never raises.
    """
    code  = entry.get("output", "")
    instr = entry.get("input", "")
    entry = dict(entry)
    entry["metadata"] = dict(entry["metadata"])

    # Prompt metadata (always computable)
    entry["metadata"]["prompt_word_count"]   = len(instr.split())
    entry["metadata"]["prompt_length_chars"] = len(instr)
    entry["metadata"]["code_lines"]          = len([l for l in code.splitlines() if l.strip()])
    entry["metadata"]["qiskit_version"]      = QISKIT_VERSION

    # tiktoken token count (optional; skip gracefully if not installed)
    try:
        import tiktoken
        enc = tiktoken.get_encoding("cl100k_base")
        entry["metadata"]["prompt_token_count_cl100k"] = len(enc.encode(instr))
    except Exception:
        entry["metadata"]["prompt_token_count_cl100k"] = None

    # Deprecated-API detection (text-level, no exec needed)
    dep_used, dep_patterns = detect_deprecated_api_usage(code)
    entry["metadata"]["api_deprecated_usage"]    = dep_used
    entry["metadata"]["deprecated_api_patterns"] = dep_patterns

    if not code or not code.strip():
        entry["metadata"]["circuit_stats_available"] = False
        entry["metadata"]["validation_status"]       = "exec_error"
        entry["metadata"]["validation_error_type"]   = "EmptyCode"
        entry["metadata"]["hallucination_type"]      = "runtime_semantic_failure"
        return entry

    ns, err = exec_with_timeout(code, timeout=3.0)
    qc      = extract_best_circuit(ns) if ns is not None else None
    v_status, v_err_type = classify_validation_status(err, qc is not None)

    entry["metadata"]["validation_status"]     = v_status
    entry["metadata"]["validation_error_type"] = v_err_type
    entry["metadata"]["hallucination_type"]    = classify_hallucination_type(
        v_status, v_err_type, error_text=err
    )

    if qc is None:
        entry["metadata"]["circuit_stats_available"] = False
        return entry

    # OpenQASM 3.0 export
    try:
        from qiskit import qasm3
        entry["openqasm3_code"] = qasm3.dumps(qc)
        entry["metadata"]["openqasm3_export_successful"] = True
        entry["metadata"]["openqasm3_export_error"]      = None
    except Exception as qasm_exc:
        entry["openqasm3_code"] = None
        entry["metadata"]["openqasm3_export_successful"] = False
        entry["metadata"]["openqasm3_export_error"]      = type(qasm_exc).__name__

    entry["metadata"].update(extract_circuit_stats(qc))
    return entry


print("Stats extraction and enrich_entry defined.")

## Cell 8 — I/O & Resume Cache

JSONL load/write, cache append/load/apply. Run once before any split cell.

In [ ]:
def load_jsonl(path: str) -> list:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(entries: list, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for e in entries:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")


def load_cache(path: str) -> dict:
    """Load enrich_metadata_cache.jsonl. Returns dict keyed by circuit_hash."""
    cache: dict = {}
    if not os.path.exists(path):
        return cache
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                ch  = rec.get("circuit_hash", "")
                if ch:
                    cache[ch] = rec
            except json.JSONDecodeError:
                pass
    return cache


def append_to_cache(entry: dict, path: str) -> None:
    """Append one enriched entry's computed fields to the cache file."""
    ch = entry.get("metadata", {}).get("circuit_hash", "")
    if not ch:
        return
    meta   = entry.get("metadata", {})
    record = {
        "circuit_hash":   ch,
        "openqasm3_code": entry.get("openqasm3_code"),
        "enriched_meta":  {k: meta[k] for k in _ENRICHMENT_FIELDS if k in meta},
    }
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def apply_cache_record(entry: dict, rec: dict) -> dict:
    """Patch an entry from cache without re-executing Qiskit."""
    entry = dict(entry)
    entry["metadata"] = dict(entry["metadata"])
    entry["metadata"].update(rec.get("enriched_meta", {}))
    entry["openqasm3_code"] = rec.get("openqasm3_code")
    return entry


def process_split(split_name: str, paths: dict, cache: dict) -> tuple:
    """
    Load, enrich, and write one split to a tmp file.
    Entries already in cache are restored without re-executing.
    Returns (enriched_list, n_enriched, n_skipped, n_from_cache, complexity_counts).
    """
    print(f"\n[{split_name}] Loading {paths['input']} ...", flush=True)
    entries = load_jsonl(paths["input"])
    total   = len(entries)
    print(f"[{split_name}] {total:,} entries to process.", flush=True)

    enriched_entries  = []
    n_enriched        = 0
    n_skipped         = 0
    n_from_cache      = 0
    complexity_counts: dict = {}
    t0 = time.time()

    for idx, entry in enumerate(entries):
        ch = entry.get("metadata", {}).get("circuit_hash", "")
        if ch and ch in cache:
            enriched = apply_cache_record(entry, cache[ch])
            n_from_cache += 1
        else:
            enriched = enrich_entry(entry)
            append_to_cache(enriched, CACHE_FILE)

        enriched_entries.append(enriched)

        if enriched["metadata"].get("circuit_stats_available"):
            n_enriched += 1
            cc = enriched["metadata"].get("size_class", "unknown")
            complexity_counts[cc] = complexity_counts.get(cc, 0) + 1
        else:
            n_skipped += 1

        if (idx + 1) % 500 == 0:
            elapsed = time.time() - t0
            pct     = 100 * (idx + 1) / total
            print(
                f"  [{split_name}] {idx+1:,}/{total:,} ({pct:.1f}%)  "
                f"enriched={n_enriched:,}  skipped={n_skipped:,}  "
                f"cached={n_from_cache:,}  elapsed={elapsed:.0f}s",
                flush=True,
            )

    elapsed = time.time() - t0
    print(
        f"[{split_name}] Done in {elapsed:.1f}s  "
        f"enriched={n_enriched:,}  skipped={n_skipped:,}  cached={n_from_cache:,}",
        flush=True,
    )
    print(f"[{split_name}] Writing tmp → {paths['tmp']}", flush=True)
    write_jsonl(enriched_entries, paths["tmp"])

    return enriched_entries, n_enriched, n_skipped, n_from_cache, complexity_counts


print("I/O and cache helpers defined.")

## Cell 9 — Load Resume Cache

Run this once before any split cell. It loads `enrich_metadata_cache.jsonl` so re-running any split cell skips already-processed entries.

In [ ]:
print(f"Loading resume cache from:\n  {CACHE_FILE}", flush=True)
cache = load_cache(CACHE_FILE)
print(f"Cache entries loaded : {len(cache):,}")

# Show which splits exist and how many entries are in each
print("\nSplit status:")
for name, paths in SPLITS.items():
    if os.path.exists(paths["input"]):
        n = sum(1 for _ in open(paths["input"], encoding="utf-8"))
        print(f"  {name:<12}: {n:,} entries  ✓")
    else:
        print(f"  {name:<12}: NOT FOUND  ✗")

## Cell 10 — Process `train_clean.jsonl`

Resume-safe: cached entries are restored without re-executing Qiskit.
Output written to `train_clean_enriched.jsonl` (tmp). **Do not rename yet — wait until all splits finish (Cell 13).**

In [ ]:
import time as _time
_t0_train = _time.time()

if os.path.exists(SPLITS["train"]["input"]):
    train_entries, train_enriched, train_skipped, train_cached, train_cc = process_split(
        "train", SPLITS["train"], cache
    )
    print(f"\nTrain total elapsed: {_time.time() - _t0_train:.1f}s")
    print("Size class breakdown:")
    for cc in ["trivial", "simple", "moderate", "complex", "very_complex"]:
        cnt = train_cc.get(cc, 0)
        if cnt:
            print(f"  {cc:<16} {cnt:,}")
else:
    print("train_clean.jsonl not found — skipping.")
    train_entries, train_enriched, train_skipped, train_cached, train_cc = [], 0, 0, 0, {}

## Cell 11 — Process `validation_clean.jsonl`

In [ ]:
_t0_val = _time.time()

if os.path.exists(SPLITS["validation"]["input"]):
    val_entries, val_enriched, val_skipped, val_cached, val_cc = process_split(
        "validation", SPLITS["validation"], cache
    )
    print(f"\nValidation total elapsed: {_time.time() - _t0_val:.1f}s")
    print("Size class breakdown:")
    for cc in ["trivial", "simple", "moderate", "complex", "very_complex"]:
        cnt = val_cc.get(cc, 0)
        if cnt:
            print(f"  {cc:<16} {cnt:,}")
else:
    print("validation_clean.jsonl not found — skipping.")
    val_entries, val_enriched, val_skipped, val_cached, val_cc = [], 0, 0, 0, {}

## Cell 12 — Process `test_clean.jsonl`

⚠️ Hold-out set — do not inspect outputs during development.

In [ ]:
_t0_test = _time.time()

if os.path.exists(SPLITS["test"]["input"]):
    test_entries, test_enriched, test_skipped, test_cached, test_cc = process_split(
        "test", SPLITS["test"], cache
    )
    print(f"\nTest total elapsed: {_time.time() - _t0_test:.1f}s")
else:
    print("test_clean.jsonl not found — skipping.")
    test_entries, test_enriched, test_skipped, test_cached, test_cc = [], 0, 0, 0, {}

## Cell 13 — Atomic Rename + Final Summary

Only run this cell after **all three split cells complete successfully**.
Atomically renames each `*_enriched.jsonl` tmp file back to `*_clean.jsonl`.

In [ ]:
# Rename tmp → original for each completed split
for split_name, paths in SPLITS.items():
    tmp_path  = paths["tmp"]
    orig_path = paths["input"]
    if os.path.exists(tmp_path):
        os.replace(tmp_path, orig_path)
        print(f"  Renamed: {os.path.basename(tmp_path)} → {os.path.basename(orig_path)}")
    else:
        print(f"  Skipped (tmp not found): {split_name}")

# Aggregate summary
all_results = {
    "train":      (train_enriched,  train_skipped,  train_cached,  train_cc),
    "validation": (val_enriched,    val_skipped,    val_cached,    val_cc),
    "test":       (test_enriched,   test_skipped,   test_cached,   test_cc),
}
total_enriched   = sum(v[0] for v in all_results.values())
total_skipped    = sum(v[1] for v in all_results.values())
total_from_cache = sum(v[2] for v in all_results.values())
total_entries    = total_enriched + total_skipped

combined_cc: dict = {}
for _, (_, _, _, cc_counts) in all_results.items():
    for cc, cnt in cc_counts.items():
        combined_cc[cc] = combined_cc.get(cc, 0) + cnt

print()
print("=" * 55)
print("=== ENRICHMENT SUMMARY ===")
print(f"Total entries  : {total_entries:,}")
print(f"Enriched (OK)  : {total_enriched:,}  ({100*total_enriched/max(total_entries,1):.1f}%)")
print(f"From cache     : {total_from_cache:,}  ({100*total_from_cache/max(total_entries,1):.1f}%)")
print(f"Skipped        : {total_skipped:,}  ({100*total_skipped/max(total_entries,1):.1f}%)")
print()
print("By split:")
for split_name, (n_enriched, n_skipped, n_cached, _) in all_results.items():
    n_total = n_enriched + n_skipped
    if n_total > 0:
        print(f"  {split_name:<12} enriched={n_enriched:,}  cached={n_cached:,}  skipped={n_skipped:,}  total={n_total:,}")
print()
print("By size_class (enriched entries only):")
for cc in ["trivial", "simple", "moderate", "complex", "very_complex", "unknown"]:
    cnt = combined_cc.get(cc, 0)
    if cnt:
        print(f"  {cc:<16} {cnt:,}")
print("=" * 55)
print("Next step: run enrich_circuit_family.py (Cell 34)")